# 🎓 Student Performance — Preprocessing, One-Hot Encoding & Visualization

---

**Dataset:** `Student_Performance_P.csv`  
**Rows:** 25,000 | **Columns:** 16  

### 📋 Notebook Outline
1. Install & Import Libraries
2. Load Dataset
3. Exploratory Data Analysis (EDA)
4. Preprocessing
5. One-Hot Encoding
6. Visualizations


## 📦 Step 1 — Install & Import Libraries

In [ ]:
# Install any missing libraries (Colab usually has these pre-installed)
# !pip install pandas numpy matplotlib seaborn -q

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Plot style
plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.family'] = 'DejaVu Sans'

PALETTE = ['#4C72B0','#DD8452','#55A868','#C44E52',
           '#8172B2','#937860','#DA8BC3','#8C8C8C','#CCB974','#64B5CD']

print('✅ Libraries imported successfully!')

## 📂 Step 2 — Load Dataset

> **Upload your CSV file** using the cell below, or mount Google Drive if the file is stored there.

In [ ]:
# ── Option A: Upload directly ──────────────────────────────
# from google.colab import files
# uploaded = files.upload()   # select Student_Performance_P.csv

import io
# filename = list(uploaded.keys())[0]
df = pd.read_csv("D:\SIT\Sem.4\PBL\Dataset PBL\priya dataset\Student_Performance_P.csv")

# ── Option B: Mount Google Drive (uncomment if needed) ────
# from google.colab import drive
# drive.mount('/content/drive')
# df = pd.read_csv('/content/drive/MyDrive/Student_Performance_P.csv')

print(f'✅ Dataset loaded — Shape: {df.shape}')

## 🔍 Step 3 — Exploratory Data Analysis (EDA)

In [ ]:
# First look
print('Shape:', df.shape)
df.head()

In [ ]:
# Data types
df.dtypes

In [ ]:
# Statistical summary
df.describe()

In [ ]:
# Missing values
missing = df.isnull().sum()
print('Missing Values per Column:')
print(missing[missing >= 0].to_string())

In [ ]:
# Unique values in categorical columns
cat_cols = df.select_dtypes(include='object').columns.tolist()
print(f'Categorical columns: {cat_cols}\n')
for col in cat_cols:
    print(f'{col:28}: {df[col].unique().tolist()}')

## 🔧 Step 4 — Preprocessing

In [ ]:
# ── 4.1 Drop student_id (not analytically useful) ─────────
df_clean = df.drop(columns=['student_id'])
print(f'Dropped student_id | New shape: {df_clean.shape}')

In [ ]:
# ── 4.2 Handle Missing Values ──────────────────────────────
print('Missing values after drop:')
print(df_clean.isnull().sum().to_string())

# Fill numeric nulls with median (safe default)
num_cols = df_clean.select_dtypes(include='number').columns
df_clean[num_cols] = df_clean[num_cols].fillna(df_clean[num_cols].median())

# Fill categorical nulls with mode
for col in df_clean.select_dtypes(include='object').columns:
    df_clean[col] = df_clean[col].fillna(df_clean[col].mode()[0])

print('\nMissing values after imputation:', df_clean.isnull().sum().sum())

In [ ]:
# ── 4.3 Outlier Detection (IQR Method) ────────────────────
score_cols = ['math_score', 'science_score', 'english_score',
              'overall_score', 'study_hours', 'attendance_percentage']

print(f'{"Column":<28} {"Outliers":>10}')
print('-' * 40)
for col in score_cols:
    Q1 = df_clean[col].quantile(0.25)
    Q3 = df_clean[col].quantile(0.75)
    IQR = Q3 - Q1
    n = ((df_clean[col] < Q1 - 1.5*IQR) | (df_clean[col] > Q3 + 1.5*IQR)).sum()
    print(f'{col:<28} {n:>10}')

In [ ]:
# ── 4.4 Standardise final_grade to uppercase ───────────────
df_clean['final_grade'] = df_clean['final_grade'].str.upper()
print('Grade values after standardisation:', sorted(df_clean['final_grade'].unique()))

In [ ]:
# ── 4.5 Preview cleaned dataset ───────────────────────────
print(f'Clean dataset shape: {df_clean.shape}')
df_clean.head()

## 🔢 Step 5 — One-Hot Encoding

Converting all categorical (string) columns into binary numeric columns using `pd.get_dummies()`.

In [ ]:
# Columns to encode
encode_cols = ['gender', 'school_type', 'parent_education',
               'internet_access', 'travel_time', 'extra_activities',
               'study_method', 'final_grade']

# Apply One-Hot Encoding
df_encoded = pd.get_dummies(df_clean, columns=encode_cols, drop_first=False)

print(f'Shape BEFORE encoding : {df_clean.shape}')
print(f'Shape AFTER  encoding : {df_encoded.shape}')
print(f'\nNew binary columns created: {df_encoded.shape[1] - df_clean.shape[1]}')

In [ ]:
# View new OHE columns
num_original = ['age','study_hours','attendance_percentage',
                'math_score','science_score','english_score','overall_score']
new_cols = [c for c in df_encoded.columns if c not in num_original]
print('One-Hot Encoded columns:')
for c in new_cols:
    print(' ', c)

In [ ]:
# Preview encoded dataframe
df_encoded.head()

## 📊 Step 6 — Visualizations

### 6.1 — Distribution of Overall Score

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(df_clean['overall_score'], bins=40, color=PALETTE[0], edgecolor='white', alpha=0.85)
ax.axvline(df_clean['overall_score'].mean(), color='red', linestyle='--', linewidth=1.8,
           label=f"Mean: {df_clean['overall_score'].mean():.1f}")
ax.axvline(df_clean['overall_score'].median(), color='orange', linestyle='--', linewidth=1.8,
           label=f"Median: {df_clean['overall_score'].median():.1f}")
ax.set_title('Distribution of Overall Score', fontsize=14, fontweight='bold')
ax.set_xlabel('Overall Score')
ax.set_ylabel('Frequency')
ax.legend()
plt.tight_layout()
plt.show()

### 6.2 — Final Grade Distribution

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
grade_order = ['A','B','C','D','E','F']
grade_counts = df_clean['final_grade'].value_counts().reindex(grade_order)
bars = ax.bar(grade_counts.index, grade_counts.values, color=PALETTE[:6], edgecolor='white', width=0.6)
for bar, v in zip(bars, grade_counts.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 40,
            str(v), ha='center', va='bottom', fontsize=10)
ax.set_title('Final Grade Distribution', fontsize=14, fontweight='bold')
ax.set_xlabel('Grade')
ax.set_ylabel('Count')
plt.tight_layout()
plt.show()

### 6.3 — Gender Distribution

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
gender_counts = df_clean['gender'].value_counts()
ax.pie(gender_counts.values, labels=gender_counts.index, autopct='%1.1f%%',
       colors=PALETTE[:3], startangle=90, wedgeprops=dict(edgecolor='white', linewidth=1.5))
ax.set_title('Gender Distribution', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

### 6.4 — Subject Score Distribution (Box Plots)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
bp = ax.boxplot([df_clean['math_score'], df_clean['science_score'], df_clean['english_score']],
                patch_artist=True, notch=True,
                medianprops=dict(color='black', linewidth=2))
for patch, color in zip(bp['boxes'], PALETTE):
    patch.set_facecolor(color)
    patch.set_alpha(0.75)
ax.set_xticklabels(['Math', 'Science', 'English'], fontsize=12)
ax.set_title('Score Distribution by Subject', fontsize=14, fontweight='bold')
ax.set_ylabel('Score')
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

### 6.5 — Study Hours vs Overall Score

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
ax.scatter(df_clean['study_hours'], df_clean['overall_score'],
           alpha=0.15, s=8, c=PALETTE[0])
m, b = np.polyfit(df_clean['study_hours'], df_clean['overall_score'], 1)
x_line = np.linspace(df_clean['study_hours'].min(), df_clean['study_hours'].max(), 300)
ax.plot(x_line, m*x_line + b, color='red', linewidth=2, label=f'Trend line (slope={m:.2f})')
ax.set_title('Study Hours vs Overall Score', fontsize=14, fontweight='bold')
ax.set_xlabel('Study Hours')
ax.set_ylabel('Overall Score')
ax.legend()
plt.tight_layout()
plt.show()

### 6.6 — Attendance % vs Overall Score

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
ax.scatter(df_clean['attendance_percentage'], df_clean['overall_score'],
           alpha=0.15, s=8, c=PALETTE[2])
m2, b2 = np.polyfit(df_clean['attendance_percentage'], df_clean['overall_score'], 1)
x2 = np.linspace(df_clean['attendance_percentage'].min(), df_clean['attendance_percentage'].max(), 300)
ax.plot(x2, m2*x2 + b2, color='red', linewidth=2, label=f'Trend line (slope={m2:.2f})')
ax.set_title('Attendance % vs Overall Score', fontsize=14, fontweight='bold')
ax.set_xlabel('Attendance Percentage')
ax.set_ylabel('Overall Score')
ax.legend()
plt.tight_layout()
plt.show()

### 6.7 — Avg Score by Parent Education

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
edu_order = ['no formal','high school','diploma','graduate','post graduate','phd']
edu_avg = df_clean.groupby('parent_education')['overall_score'].mean().reindex(edu_order)
bars = ax.barh(edu_avg.index, edu_avg.values, color=PALETTE[:6], edgecolor='white')
ax.axvline(df_clean['overall_score'].mean(), color='red', linestyle='--', linewidth=1.5, label='Overall Mean')
for bar, v in zip(bars, edu_avg.values):
    ax.text(v + 0.2, bar.get_y() + bar.get_height()/2, f'{v:.1f}', va='center', fontsize=9)
ax.set_title('Avg Score by Parent Education', fontsize=14, fontweight='bold')
ax.set_xlabel('Average Overall Score')
ax.legend()
plt.tight_layout()
plt.show()

### 6.8 — Avg Score by Study Method

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
sm_avg = df_clean.groupby('study_method')['overall_score'].mean().sort_values(ascending=False)
bars = ax.bar(sm_avg.index, sm_avg.values, color=PALETTE[:len(sm_avg)], edgecolor='white')
for bar, v in zip(bars, sm_avg.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.2,
            f'{v:.1f}', ha='center', va='bottom', fontsize=10)
ax.set_title('Avg Score by Study Method', fontsize=14, fontweight='bold')
ax.set_xlabel('Study Method')
ax.set_ylabel('Average Overall Score')
ax.tick_params(axis='x', rotation=20)
plt.tight_layout()
plt.show()

### 6.9 — Subject Scores by School Type

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
school_scores = df_clean.groupby('school_type')[['math_score','science_score','english_score']].mean()
x = np.arange(len(school_scores.index))
w = 0.25
for i, (col, color) in enumerate(zip(['math_score','science_score','english_score'], PALETTE)):
    ax.bar(x + i*w, school_scores[col], w,
           label=col.replace('_score','').title(), color=color, edgecolor='white')
ax.set_xticks(x + w)
ax.set_xticklabels(school_scores.index, fontsize=12)
ax.set_title('Avg Subject Scores by School Type', fontsize=14, fontweight='bold')
ax.set_ylabel('Average Score')
ax.legend()
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

### 6.10 — Grade Distribution by Internet Access (Stacked Bar)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
pivot = df_clean.groupby(['internet_access','final_grade']).size().unstack(fill_value=0)
pivot = pivot.reindex(columns=['A','B','C','D','E','F'])
pivot.plot(kind='bar', ax=ax, stacked=True, color=PALETTE[:6],
           edgecolor='white', width=0.5)
ax.set_title('Grade Distribution by Internet Access', fontsize=14, fontweight='bold')
ax.set_xlabel('Internet Access')
ax.set_ylabel('Count')
ax.tick_params(axis='x', rotation=0)
ax.legend(title='Grade', bbox_to_anchor=(1.01, 1), loc='upper left')
plt.tight_layout()
plt.show()

### 6.11 — OHE Feature Correlation with Overall Score

In [ ]:
num_original = ['age','study_hours','attendance_percentage',
                'math_score','science_score','english_score','overall_score']
ohe_bool_cols = [c for c in df_encoded.columns if c not in num_original]

corr_data = df_encoded[ohe_bool_cols + ['overall_score']].corr()['overall_score'].drop('overall_score')
corr_data = corr_data.sort_values()

fig, ax = plt.subplots(figsize=(10, 8))
colors = ['#C44E52' if v < 0 else '#4C72B0' for v in corr_data.values]
ax.barh(corr_data.index, corr_data.values, color=colors, edgecolor='white')
ax.axvline(0, color='black', linewidth=0.8)
ax.set_title('Correlation of One-Hot Encoded Features\nwith Overall Score',
             fontsize=14, fontweight='bold')
ax.set_xlabel('Pearson Correlation Coefficient')
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

## ✅ Summary

| Step | Action | Result |
|------|--------|--------|
| Load | Read CSV | 25,000 × 16 |
| Drop | Removed `student_id` | 25,000 × 15 |
| Nulls | Checked & imputed | 0 missing values |
| Outliers | IQR detection | ~24 minor outliers (retained) |
| Standardise | Uppercase `final_grade` | e→E, d→D ... |
| OHE | 8 categorical columns encoded | 25,000 × 38 |
| Visualizations | 11 matplotlib charts | Overview + Insights |
